# 49W — Scrape YouTube Transcripts
Fetches all video transcripts from the 49W channel and saves them to Google Drive.

**No YouTube API key needed.**

In [ ]:
!pip install -q yt-dlp youtube-transcript-api tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CONFIG ─────────────────────────────────────────────────
CHANNEL_URL = "https://www.youtube.com/@49W"
OUTPUT_DIR  = "/content/drive/MyDrive/49w-bot/data/raw/transcripts"
LANGUAGES   = ["tr", "en"]
# ────────────────────────────────────────────────────────────

In [ ]:
import json, time, subprocess
from pathlib import Path
from tqdm.notebook import tqdm
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Use yt-dlp to get all video IDs and titles (no API key needed)
print(f"Fetching video list from: {CHANNEL_URL}")
result = subprocess.run(
    ["yt-dlp", "--flat-playlist", "--print", "%(id)s\t%(title)s", f"{CHANNEL_URL}/videos"],
    capture_output=True, text=True
)

video_list = []
for line in result.stdout.strip().splitlines():
    parts = line.split("\t", 1)
    if len(parts) == 2:
        video_list.append({"video_id": parts[0], "title": parts[1]})

print(f"Found {len(video_list)} videos")

# Save video list
videos_path = Path(OUTPUT_DIR).parent / "videos.json"
with open(videos_path, "w", encoding="utf-8") as f:
    json.dump(video_list, f, ensure_ascii=False, indent=2)
print(f"Saved to {videos_path}")

In [ ]:
def fetch_transcript(video_id):
    try:
        fetched = YouTubeTranscriptApi().fetch(video_id, languages=LANGUAGES)
        return [{"text": s.text, "start": s.start, "duration": s.duration} for s in fetched]
    except (NoTranscriptFound, TranscriptsDisabled):
        return None
    except Exception as e:
        print(f"  Error {video_id}: {e}")
        return None

def transcript_to_text(transcript):
    return " ".join(seg["text"].strip() for seg in transcript)

fetched, skipped, failed = 0, 0, 0

for video in tqdm(video_list, desc="Fetching transcripts"):
    vid_id = video["video_id"]
    out_file = Path(OUTPUT_DIR) / f"{vid_id}.json"

    if out_file.exists():
        skipped += 1
        continue

    transcript = fetch_transcript(vid_id)
    if transcript is None:
        failed += 1
        continue

    record = {
        "video_id": vid_id,
        "title": video["title"],
        "transcript_raw": transcript,
        "transcript_text": transcript_to_text(transcript),
    }
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2)
    fetched += 1
    time.sleep(0.3)

print(f"\nDone! Fetched: {fetched} | Cached: {skipped} | No transcript: {failed}")

In [ ]:
files = list(Path(OUTPUT_DIR).glob("*.json"))
total_words = sum(len(json.load(open(f))["transcript_text"].split()) for f in files)
print(f"Transcripts saved : {len(files)}")
print(f"Total words       : {total_words:,}")
print(f"Avg words/video   : {total_words // max(len(files), 1):,}")
print(f"Location          : {OUTPUT_DIR}")